<!-- notebook-header -->
# NLP Classico e Pre-processamento de Texto

**Modulo:** 05 - Dominios Aplicados / 05B - NLP  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Tokenizacao, normalizacao, bag-of-words, TF-IDF, n-grams e classificacao classica.


# 5B_1: NLP Classico e Pre-processamento de Texto

## Visao Geral

Neste notebook, exploraremos os fundamentos de Natural Language Processing (NLP),
desde pre-processamento de texto ate representacoes classicas como Bag of Words e TF-IDF.
Estes conceitos sao a base para tudo que vem depois (embeddings, RNNs, Transformers, LLMs).

**Conteudo:**
1. Fundamentos de NLP
2. Pre-processamento de Texto
3. Representacoes Sparse (BoW, TF-IDF)
4. N-gramas e Modelos de Linguagem
5. Classificacao de Texto
6. Analise de Sentimento
7. Exercicios Praticos
8. Erros Comuns

**Pre-requisitos:** Modulo 1 (fundamentos de ML), Modulo 2 (modelos classicos)

**Dependencias:** numpy, matplotlib, collections, re

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter, defaultdict
import math
print('Imports OK')

## 1. Fundamentos de NLP

### Analogia: Ensinar um Computador a Ler

Imagine ensinar uma crianca a ler. Primeiro, ela aprende:
1. **Letras** (caracteres)
2. **Palavras** (tokens)
3. **Frases** (sequencias)
4. **Significado** (semantica)

NLP segue o mesmo caminho, mas para maquinas. O desafio e que linguagem
natural e AMBIGUA, CONTEXTUAL e VARIAVEL de formas que numeros nao sao.

### Por que em ML texto e tao diferente de numeros

Dados tabulares tem features numericas claras (idade=25, peso=70).
Texto NAO tem isso. "O banco estava cheio" -- banco de sentar ou banco financeiro?
Antes de qualquer modelo, precisamos CONVERTER texto em numeros.

### Niveis de Analise em NLP

| Nivel | O que analisa | Exemplo |
|-------|--------------|---------|
| Lexico | Palavras individuais | tokenizacao, stemming |
| Sintatico | Estrutura gramatical | parsing, POS tagging |
| Semantico | Significado | word embeddings, NER |
| Pragmatico | Contexto e intencao | sentiment, ironia |

### O que observar sobre a Pipeline de NLP

Toda tarefa de NLP segue uma pipeline: Texto -> Pre-processamento -> Representacao
-> Modelo -> Predicao. O pre-processamento e CRITICO -- garbage in, garbage out.
Um bom pre-processamento pode importar mais que a escolha do modelo.

### O que concluir sobre NLP Classico vs Deep Learning

NLP classico (BoW, TF-IDF, regex) funciona surpreendentemente bem para muitas tarefas:
classificacao simples de texto, deteccao de spam, busca por keywords.
Deep learning (BERT, GPT) e necessario para tarefas que exigem ENTENDIMENTO
de contexto: traducao, sumarizacao, QA.

### Conexao com outros notebooks sobre Representacao de Dados

A conversao de texto para numeros e analoga ao feature engineering de 1_3.
BoW e TF-IDF sao metodos de feature extraction, assim como PCA (3_2)
extrai features de dados numericos.

In [ ]:

# Exemplo de pipeline básico de NLP
class SimplePipeline:
    def __init__(self):
        self.raw_text = None
        self.tokens = None
        self.cleaned_tokens = None
    
    def load_text(self, text):
        self.raw_text = text
        return self
    
    def tokenize(self):
        # Simple whitespace tokenization
        self.tokens = self.raw_text.lower().split()
        return self
    
    def clean(self, remove_punctuation=True):
        cleaned = []
        for token in self.tokens:
            if remove_punctuation:
                token = re.sub(r'[^\w]', '', token)
            if token:
                cleaned.append(token)
        self.cleaned_tokens = cleaned
        return self

# Test pipeline
pipeline = SimplePipeline()
pipeline.load_text("O NLP é fascinante! Processamos linguagem natural.").tokenize().clean()
print(f"Tokens originais: {pipeline.tokens}")
print(f"Tokens limpos: {pipeline.cleaned_tokens}")


## 2. Pre-processamento de Texto

### Analogia: Limpando uma Cozinha Antes de Cozinhar

Antes de cozinhar, voce limpa a bancada, separa ingredientes, e organiza tudo.
Pre-processamento de texto e a mesma coisa:
- **Remover lixo** (HTML tags, caracteres especiais)
- **Normalizar** (minusculas, acentos)
- **Segmentar** (dividir em tokens)
- **Filtrar** (remover stopwords)
- **Reduzir** (stemming/lemmatization)

### Por que em ML cada etapa importa

Cada etapa de pre-processamento REDUZ o vocabulario, facilitando o aprendizado:
- Lowercase: "Casa" e "casa" viram a mesma feature
- Stopwords: remove palavras sem conteudo semantico
- Stemming: "correndo", "correr", "corrida" viram "corr"

Menos features unicas = menos esparsidade = modelos melhores.

### O que observar sobre a ordem do pre-processamento

A ORDEM importa! Exemplo:
1. Se voce remove pontuacao ANTES de tokenizar, "nao." vira "nao" (correto)
2. Se tokeniza primeiro, "nao." e um token diferente de "nao" (problema)

Pipeline recomendada: lowercase -> remover HTML -> remover pontuacao -> tokenizar
-> remover stopwords -> stemming.

In [ ]:

# Tokenização em diferentes níveis
text = "Dr. Smith trabalhou na empresa. Ele ganhou $100 em 2024!"

# Tokenização simples
simple_tokens = text.split()
print(f"Tokenização simples: {simple_tokens}")

# Tokenização com regex
regex_tokens = re.findall(r"\b\w+(?:'\w+)?\b|\$\d+", text)
print(f"Tokenização regex: {regex_tokens}")

# Tokenização com pontuação
punct_pattern = r"\w+|[.,!?;:]"
punct_tokens = re.findall(punct_pattern, text)
print(f"Com pontuação: {punct_tokens}")


In [ ]:

# Stopwords em português
stopwords_pt = {
    'o', 'a', 'os', 'as', 'um', 'uma', 'uns', 'umas',
    'de', 'do', 'da', 'dos', 'das', 'em', 'no', 'na',
    'e', 'ou', 'mas', 'porém', 'que', 'qual', 'quais',
    'é', 'são', 'foi', 'foram', 'ser', 'estar',
    'para', 'por', 'com', 'sem', 'entre'
}

text_tokens = ['o', 'nlp', 'é', 'fascinante', 'e', 'importante']
filtered = [t for t in text_tokens if t not in stopwords_pt]
print(f"Original: {text_tokens}")
print(f"Sem stopwords: {filtered}")


In [ ]:

# Stemming simples - remover sufixos comuns
def simple_stemming(word):
    # Sufixos portugueses comuns
    suffixes = ['mente', 'ação', 'ação', 'ador', 'ável', 'ível', 'ismo', 'ista', 'ismo']
    for suffix in suffixes:
        if word.endswith(suffix):
            return word[:-len(suffix)]
    # Plural
    if word.endswith('s'):
        return word[:-1]
    return word

words = ['correr', 'correndo', 'corredor', 'funcionamento', 'funcional', 'gatos']
stemmed = [(w, simple_stemming(w)) for w in words]
print("Stemming (palavra -> raiz):")
for word, stem in stemmed:
    print(f"  {word:15} -> {stem}")


In [ ]:

# Normalização de texto
def normalize_text(text):
    # Converter para lowercase
    text = text.lower()
    # Remover acentos
    text = text.replace('á', 'a').replace('é', 'e').replace('í', 'i')
    text = text.replace('ó', 'o').replace('ú', 'u').replace('ã', 'a')
    text = text.replace('õ', 'o').replace('ç', 'c')
    # Remover espaços extras
    text = ' '.join(text.split())
    return text

text = "  A Programação é IMPORTANTE! Não deixe para depois...  "
normalized = normalize_text(text)
print(f"Original: '{text}'")
print(f"Normalizado: '{normalized}'")


### O que concluir sobre Pre-processamento

Pre-processamento e a parte MENOS glamurosa de NLP, mas a MAIS importante
para modelos classicos. Um BoW com bom pre-processamento frequentemente
supera um BoW com pre-processamento ruim, mesmo com modelo mais sofisticado.

### Conexao com outros notebooks sobre Feature Engineering

Pre-processamento de texto e essencialmente feature engineering (1_3) para texto.
Cada decisao (remover stopwords? usar stemming?) muda as features que o modelo ve.
Assim como normalizar features numericas ajuda modelos (4_1), normalizar texto ajuda NLP.

## 3. Representacoes Sparse: Bag of Words e TF-IDF

### Analogia: Bag of Words como Sacola de Supermercado

Bag of Words trata um texto como uma sacola de compras:
- Voce sabe O QUE esta na sacola (quais palavras)
- Voce sabe QUANTO de cada item (frequencia)
- Voce NAO sabe a ORDEM (posicao das palavras)

"O gato comeu o rato" e "O rato comeu o gato" tem o MESMO BoW!

### Por que em ML BoW funciona apesar de ignorar ordem

Para muitas tarefas, a PRESENCA de palavras e mais informativa que a ordem:
- Classificacao de spam: "gratis", "promocao", "clique" indicam spam
- Analise de sentimento: "otimo", "pessimo", "maravilhoso" indicam tom
- Categorizacao de topicos: "gol", "jogador", "campeonato" indicam esportes

### TF-IDF: Ponderando Palavras por Importancia

BoW trata todas as palavras igual. Mas "o" aparece em TAREFA DO ALUNO texto (pouco informativo).
TF-IDF pondera: TF (frequencia no documento) x IDF (raridade no corpus).

**Intuicao:** Uma palavra que aparece MUITO em um documento mas POUCO em outros
e muito informativa sobre aquele documento.

### O que observar sobre a Dimensionalidade

BoW cria uma feature por palavra unica. Um corpus com 100K palavras unicas
gera vetores de 100K dimensoes -- a maioria zeros (sparse).
Isso e o "curse of dimensionality" aplicado a texto.

### O que concluir sobre as Limitacoes de BoW/TF-IDF

BoW/TF-IDF NAO capturam:
- Sinonimos ("feliz" e "contente" sao features diferentes)
- Polissemia ("banco" financeiro vs "banco" de praca)
- Ordem ("nao gostei" vs "gostei" se nao usar n-grams)
- Contexto de longa distancia

Word embeddings (5B_2) resolvem esses problemas.

In [ ]:

# Implementação de Bag of Words
class BagOfWords:
    def __init__(self):
        self.vocab = {}
        self.vocab_index = 0
    
    def fit(self, texts):
        # Build vocabulary
        for text in texts:
            words = text.lower().split()
            for word in words:
                if word not in self.vocab:
                    self.vocab[word] = self.vocab_index
                    self.vocab_index += 1
        return self
    
    def transform(self, text):
        # Create BoW vector
        words = text.lower().split()
        vector = np.zeros(len(self.vocab))
        for word in words:
            if word in self.vocab:
                vector[self.vocab[word]] += 1
        return vector
    
    def fit_transform(self, texts):
        self.fit(texts)
        return np.array([self.transform(text) for text in texts])

# Test BoW
texts = ['gato subiu na árvore', 'cachorro correu no parque', 'gato dormiu']
bow = BagOfWords()
matrix = bow.fit_transform(texts)
print(f"Vocabulário: {bow.vocab}")
print(f"\nMatriz BoW:\n{matrix}")
print(f"\nDimensões: {matrix.shape}")


In [ ]:

# Implementação de TF-IDF
class TfidfVectorizer:
    def __init__(self):
        self.vocab = {}
        self.idf = {}
    
    def fit(self, texts):
        # Build vocabulary
        vocab_set = set()
        doc_freq = defaultdict(int)
        
        for text in texts:
            words = set(text.lower().split())
            vocab_set.update(words)
            for word in words:
                doc_freq[word] += 1
        
        self.vocab = {word: idx for idx, word in enumerate(sorted(vocab_set))}
        
        # Calculate IDF: log(num_docs / doc_freq)
        num_docs = len(texts)
        for word in self.vocab:
            self.idf[word] = math.log(num_docs / doc_freq[word])
        
        return self
    
    def transform(self, text):
        vector = np.zeros(len(self.vocab))
        words = text.lower().split()
        
        # Count term frequency
        tf = Counter(words)
        
        # Calculate TF-IDF
        for word, freq in tf.items():
            if word in self.vocab:
                idx = self.vocab[word]
                vector[idx] = freq * self.idf[word]
        
        return vector

# Test TF-IDF
texts = ['gato subiu na árvore', 'cachorro correu no parque', 'gato dormiu']
tfidf = TfidfVectorizer()
tfidf.fit(texts)
matrix = np.array([tfidf.transform(text) for text in texts])
print(f"TF-IDF matrix shape: {matrix.shape}")
print(f"TF-IDF primeira linha:\n{matrix[0]}")


In [ ]:
# Visualizacao: BoW vs TF-IDF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Corpus de exemplo
docs = [
    "machine learning e inteligencia artificial",
    "deep learning usa redes neurais para machine learning",
    "processamento de linguagem natural usa machine learning",
    "o gato sentou no tapete"
]

# BoW
vocab = set()
for doc in docs:
    vocab.update(doc.lower().split())
vocab = sorted(vocab)

bow_matrix = np.zeros((len(docs), len(vocab)))
for i, doc in enumerate(docs):
    words = doc.lower().split()
    for w in words:
        j = vocab.index(w)
        bow_matrix[i, j] += 1

# TF-IDF
tf = bow_matrix / (bow_matrix.sum(axis=1, keepdims=True) + 1e-8)
df = (bow_matrix > 0).sum(axis=0)
idf = np.log(len(docs) / (df + 1))
tfidf_matrix = tf * idf

# Plot BoW
ax = axes[0]
im = ax.imshow(bow_matrix, cmap='YlOrRd', aspect='auto')
ax.set_title('Bag of Words\n(frequencia bruta)', fontsize=11, fontweight='bold')
ax.set_yticks(range(len(docs)))
ax.set_yticklabels([f'Doc {i}' for i in range(len(docs))], fontsize=9)
ax.set_xticks(range(len(vocab)))
ax.set_xticklabels(vocab, rotation=45, ha='right', fontsize=7)
for i in range(bow_matrix.shape[0]):
    for j in range(bow_matrix.shape[1]):
        if bow_matrix[i, j] > 0:
            ax.text(j, i, f'{int(bow_matrix[i,j])}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.8)

# Plot TF-IDF
ax = axes[1]
im = ax.imshow(tfidf_matrix, cmap='YlOrRd', aspect='auto')
ax.set_title('TF-IDF\n(frequencia x raridade)', fontsize=11, fontweight='bold')
ax.set_yticks(range(len(docs)))
ax.set_yticklabels([f'Doc {i}' for i in range(len(docs))], fontsize=9)
ax.set_xticks(range(len(vocab)))
ax.set_xticklabels(vocab, rotation=45, ha='right', fontsize=7)
for i in range(tfidf_matrix.shape[0]):
    for j in range(tfidf_matrix.shape[1]):
        if tfidf_matrix[i, j] > 0.01:
            ax.text(j, i, f'{tfidf_matrix[i,j]:.2f}', ha='center', va='center', fontsize=6)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('/tmp/bow_vs_tfidf.png', dpi=100, bbox_inches='tight')
plt.show()

# Comparar: "machine" aparece em 3/4 docs
idx_ml = vocab.index('machine')
print(f'Palavra "machine":')
print(f'  BoW: aparece {int(bow_matrix[:, idx_ml].sum())} vezes total')
print(f'  IDF: {idf[idx_ml]:.3f} (baixo -- pouco informativa)')

# "gato" aparece em 1/4 docs
idx_gato = vocab.index('gato')
print(f'Palavra "gato":')
print(f'  BoW: aparece {int(bow_matrix[:, idx_gato].sum())} vezes total')
print(f'  IDF: {idf[idx_gato]:.3f} (alto -- muito informativa)')
print()
print('TF-IDF: palavras RARAS em outros docs mas FREQUENTES neste doc recebem peso alto')

## 4. N-gramas e Modelos de Linguagem

### Por que em ML n-gramas resolvem o problema de ordem

BoW ignora ordem. N-gramas capturam sequencias de N palavras:
- Unigrama: ["nao", "gostei"] (perde a negacao)
- Bigrama: ["nao_gostei"] (captura a negacao!)
- Trigrama: ["eu_nao_gostei"] (mais contexto)

### O que observar sobre o trade-off de N

| N | Vantagem | Desvantagem |
|---|----------|-------------|
| 1 | Vocabulario pequeno | Perde ordem |
| 2 | Captura pares | Vocabulario V^2 |
| 3 | Mais contexto | Vocabulario V^3 (explode!) |
| 4+ | Muita esparsidade | Quase nao aparece no corpus |

Na pratica, uni+bigramas sao o sweet spot para a maioria das tarefas classicas.

### Conexao com outros notebooks sobre Dimensionalidade

O problema de n-gramas com N grande conecta com 3_2 (reducao de dimensionalidade).
Com N=3, o vocabulario pode ter milhoes de features, a maioria com contagem zero.

In [ ]:

# N-gramas
def create_ngrams(text, n=2):
    words = text.lower().split()
    ngrams = []
    for i in range(len(words) - n + 1):
        ngrams.append(' '.join(words[i:i+n]))
    return ngrams

text = "o gato subiu na árvore grande"
print(f"Texto: {text}")
print(f"\nUnigramas (n=1): {create_ngrams(text, 1)}")
print(f"Bigramas (n=2): {create_ngrams(text, 2)}")
print(f"Trigramas (n=3): {create_ngrams(text, 3)}")

# Frequência de N-gramas
bigrams = create_ngrams(text, 2)
freq = Counter(bigrams)
print(f"\nFrequência de bigramas: {dict(freq)}")


## 5. Classificacao de Texto

### Analogia: Separar Cartas por Assunto

Classificacao de texto e como separar cartas em pilhas por assunto:
leia as palavras-chave e decida a categoria. NLP classico faz exatamente isso
usando BoW/TF-IDF como features + um classificador.

### Por que em ML Naive Bayes funciona tao bem para texto

Naive Bayes assume independencia entre features (palavras). Isso e FALSO
para texto (palavras sao dependentes). Mas funciona bem porque:
1. Com features de alta dimensao, a independencia e menos prejudicial
2. Os ERROS se cancelam entre features
3. E rapido e nao precisa de muitos dados

### O que observar sobre o Pipeline de Classificacao

1. Pre-processamento (limpar, tokenizar)
2. Vetorizacao (BoW ou TF-IDF)
3. Classificador (Naive Bayes, Logistic Regression, SVM)
4. Avaliacao (accuracy, F1, confusion matrix)

### O que concluir sobre quando usar NLP Classico vs Deep Learning

| Cenario | Recomendacao | Por que |
|---------|-------------|---------|
| <1K exemplos | NLP classico | Pouco dado, deep learning overfita |
| 1K-100K exemplos | Teste ambos | Depende da complexidade |
| >100K exemplos | Deep learning | Dados suficientes para BERT/GPT |
| Precisa de interpretabilidade | NLP classico | Features sao palavras compreensiveis |
| Real-time, baixa latencia | NLP classico | Inferencia rapida |

### Conexao com outros notebooks sobre Classificacao

Classificacao de texto usa os MESMOS classificadores de 2_1 (Naive Bayes),
2_2 (Logistic Regression), 2_3 (SVM). A unica diferenca e a feature extraction.

In [ ]:
# Classificador Naive Bayes para texto (implementacao manual)
class NaiveBayesText:
    def __init__(self):
        self.class_priors = {}
        self.word_probs = {}
        self.vocab = set()

    def fit(self, texts, labels):
        # Contar classes
        label_counts = Counter(labels)
        total = len(labels)
        self.class_priors = {c: count/total for c, count in label_counts.items()}

        # Contar palavras por classe
        word_counts = defaultdict(lambda: defaultdict(int))
        class_totals = defaultdict(int)

        for text, label in zip(texts, labels):
            words = text.lower().split()
            for word in words:
                self.vocab.add(word)
                word_counts[label][word] += 1
                class_totals[label] += 1

        # Calcular probabilidades com Laplace smoothing
        V = len(self.vocab)
        self.word_probs = {}
        for c in label_counts:
            self.word_probs[c] = {}
            for word in self.vocab:
                count = word_counts[c].get(word, 0)
                self.word_probs[c][word] = (count + 1) / (class_totals[c] + V)

    def predict(self, text):
        words = text.lower().split()
        scores = {}
        for c in self.class_priors:
            score = math.log(self.class_priors[c])
            for word in words:
                if word in self.vocab:
                    score += math.log(self.word_probs[c].get(word, 1/len(self.vocab)))
            scores[c] = score
        return max(scores, key=scores.get), scores

# Dataset de exemplo
train_texts = [
    "otimo produto recomendo muito bom",
    "excelente qualidade adorei perfeito",
    "maravilhoso funciona super bem",
    "produto incrivel satisfeito compra",
    "pessimo nao funciona horrivel",
    "terrivel qualidade ruim decepcionado",
    "nao recomendo pior produto",
    "defeituoso lixo nao presta",
]
train_labels = ['positivo'] * 4 + ['negativo'] * 4

nb_model = NaiveBayesText()
nb_model.fit(train_texts, train_labels)

# Testar
test_texts = [
    "produto bom recomendo",
    "pessimo nao gostei",
    "qualidade excelente adorei",
    "horrivel defeituoso terrivel"
]

print('Classificacao Naive Bayes para Sentimento')
print('=' * 50)
for text in test_texts:
    pred, scores = nb_model.predict(text)
    print(f'  "{text}"')
    print(f'    -> {pred} (log-scores: pos={scores["positivo"]:.2f}, neg={scores["negativo"]:.2f})')
print()
print(f'Vocabulario: {len(nb_model.vocab)} palavras unicas')
print(f'Priors: {nb_model.class_priors}')

## 6. Analise de Sentimento

### Por que em ML sentimento e uma tarefa fundamental

Analise de sentimento e o "hello world" de NLP -- simples o suficiente para
ensinar conceitos, mas util o suficiente para ter aplicacoes reais:
- Monitorar opiniao de clientes em redes sociais
- Avaliar reviews de produtos
- Medir satisfacao em pesquisas

### O que observar sobre as Abordagens

1. **Baseada em lexico:** Dicionario de palavras positivas/negativas. Simples, sem treino.
2. **Baseada em ML:** BoW/TF-IDF + classificador. Precisa de dados rotulados.
3. **Deep Learning:** BERT fine-tuned. Melhor accuracy, mais caro.

### O que concluir sobre Desafios de Sentimento

- **Negacao:** "nao e ruim" = positivo (dificil para BoW)
- **Ironia:** "que maravilha, meu celular quebrou" = negativo
- **Dominio:** "acido" e negativo em reviews mas neutro em quimica
- **Intensidade:** "bom" vs "excelente" vs "perfeito" -- graus diferentes

In [ ]:

# Análise de sentimento simples baseada em léxico
positive_words = {'bom', 'ótimo', 'excelente', 'adorei', 'perfeito', 'incrível'}
negative_words = {'ruim', 'horrível', 'péssimo', 'odiei', 'terrível', 'chato'}

def simple_sentiment(text):
    words = set(text.lower().split())
    pos_count = len(words & positive_words)
    neg_count = len(words & negative_words)
    
    if pos_count > neg_count:
        return 'positivo', pos_count - neg_count
    elif neg_count > pos_count:
        return 'negativo', neg_count - pos_count
    else:
        return 'neutro', 0

# Test
test_sentences = [
    'Este filme é excelente e perfeito!',
    'Que horrível e ruim!',
    'O livro é bom.',
    'Não gostei do filme.'
]

print("Análise de Sentimento (léxico):")
for sent in test_sentences:
    sentiment, score = simple_sentiment(sent)
    print(f"  '{sent}' -> {sentiment} (score: {score})")


In [ ]:
# Similaridade entre documentos usando TF-IDF
def cosine_similarity(a, b):
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return dot / (norm + 1e-8)

# Corpus
docs = [
    "inteligencia artificial e machine learning transformam negocios",
    "deep learning e uma tecnica de machine learning com redes neurais",
    "o mercado financeiro brasileiro cresceu no ultimo trimestre",
    "acoes e investimentos em bolsa sao populares no brasil",
    "processamento de linguagem natural usa tecnicas de machine learning",
]

# Construir TF-IDF
all_words = set()
for doc in docs:
    all_words.update(doc.lower().split())
vocab_list = sorted(all_words)
V = len(vocab_list)
word_to_idx = {w: i for i, w in enumerate(vocab_list)}

# TF
tf_matrix = np.zeros((len(docs), V))
for i, doc in enumerate(docs):
    words = doc.lower().split()
    for w in words:
        tf_matrix[i, word_to_idx[w]] += 1
    tf_matrix[i] /= (len(words) + 1e-8)

# IDF
df = (tf_matrix > 0).sum(axis=0)
idf_vec = np.log(len(docs) / (df + 1))

# TF-IDF
tfidf = tf_matrix * idf_vec

# Matriz de similaridade
n = len(docs)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(tfidf[i], tfidf[j])

# Visualizar
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='YlGnBu', vmin=0, vmax=1)
ax.set_title('Similaridade Cosseno entre Documentos (TF-IDF)', fontsize=12, fontweight='bold')
labels = [f'Doc {i}: {docs[i][:25]}...' for i in range(n)]
ax.set_xticks(range(n))
ax.set_xticklabels([f'D{i}' for i in range(n)])
ax.set_yticks(range(n))
ax.set_yticklabels(labels, fontsize=8)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('/tmp/doc_similarity.png', dpi=100, bbox_inches='tight')
plt.show()

print('Documentos sobre ML (0,1,4) sao similares entre si')
print('Documentos sobre financas (2,3) sao similares entre si')
print('Cross-topico: similaridade baixa')

### Conexao com outros notebooks sobre Metricas de Distancia

Similaridade cosseno e uma metrica de distancia assim como euclidiana (1_2).
Para texto sparse (BoW/TF-IDF), cosseno e MELHOR que euclidiana porque
ignora a magnitude (documentos longos vs curtos) e foca na direcao.

## 7. Exercicios Praticos

### Exercicio 1: Pipeline de Pre-processamento

**Tarefa:** Implemente uma funcao que aplica o pipeline completo de pre-processamento:
lowercase, remover pontuacao, tokenizar, remover stopwords, e contar frequencias.

In [ ]:
# PRATICA - Exercicio 1: Pipeline de Pre-processamento
# Implemente a funcao preprocess_text

stopwords_pt = {'o', 'a', 'os', 'as', 'um', 'uma', 'de', 'do', 'da', 'em', 'no', 'na',
                'e', 'que', 'para', 'com', 'por', 'se', 'nao', 'mais', 'como', 'mas',
                'ao', 'seu', 'sua', 'ou', 'ser', 'quando', 'muito', 'ja', 'tambem',
                'so', 'pelo', 'pela', 'ate', 'isso', 'ela', 'entre', 'era', 'depois',
                'este', 'esta', 'nos', 'voce', 'foi', 'sao', 'tem', 'ha', 'ele'}

texto = "O Machine Learning, que e uma area da Inteligencia Artificial, tem revolucionado a forma como processamos dados! Nao e so para especialistas."

def preprocess_text(text, stop_words):
    # TAREFA DO ALUNO: 1. Converter para minusculas
    text_lower = None
    # TAREFA DO ALUNO: 2. Remover pontuacao (usar re.sub)
    text_clean = None
    # TAREFA DO ALUNO: 3. Tokenizar (split por espacos)
    tokens = None
    # TAREFA DO ALUNO: 4. Remover stopwords
    filtered = None
    # TAREFA DO ALUNO: 5. Contar frequencias (usar Counter)
    freq = None
    return filtered, freq

result, freq = preprocess_text(texto, stopwords_pt)
if result is not None:
    print(f'Tokens filtrados: {result}')
    print(f'Frequencias: {dict(freq)}')
else:
    print('Complete os TAREFA DO ALUNOs acima!')

In [ ]:
# SOLUCAO - Exercicio 1: Pipeline de Pre-processamento
stopwords_pt = {'o', 'a', 'os', 'as', 'um', 'uma', 'de', 'do', 'da', 'em', 'no', 'na',
                'e', 'que', 'para', 'com', 'por', 'se', 'nao', 'mais', 'como', 'mas',
                'ao', 'seu', 'sua', 'ou', 'ser', 'quando', 'muito', 'ja', 'tambem',
                'so', 'pelo', 'pela', 'ate', 'isso', 'ela', 'entre', 'era', 'depois',
                'este', 'esta', 'nos', 'voce', 'foi', 'sao', 'tem', 'ha', 'ele'}

texto = "O Machine Learning, que e uma area da Inteligencia Artificial, tem revolucionado a forma como processamos dados! Nao e so para especialistas."

def preprocess_text(text, stop_words):
    # 1. Lowercase
    text_lower = text.lower()
    # 2. Remover pontuacao
    text_clean = re.sub(r'[^a-z0-9\s]', '', text_lower)
    # 3. Tokenizar
    tokens = text_clean.split()
    # 4. Remover stopwords
    filtered = [t for t in tokens if t not in stop_words]
    # 5. Frequencias
    freq = Counter(filtered)
    return filtered, freq

result, freq = preprocess_text(texto, stopwords_pt)
print('Pipeline de Pre-processamento - SOLUCAO')
print(f'  Texto original: {texto[:50]}...')
print(f'  Tokens filtrados ({len(result)}): {result}')
print(f'  Frequencias: {dict(freq)}')
print()
print(f'  Reducao: de {len(texto.split())} palavras para {len(result)} tokens')
print(f'  Removidas: stopwords e pontuacao')

### Exercicio 2: TF-IDF Manual

**Tarefa:** Calcule TF-IDF manualmente para um corpus de 3 documentos.
Identifique qual palavra e mais informativa para cada documento.

In [ ]:
# PRATICA - Exercicio 2: TF-IDF Manual
corpus = [
    "gato gato cachorro",
    "cachorro cachorro cachorro gato",
    "passaro passaro passaro passaro"
]

# TAREFA DO ALUNO: Tokenizar cada documento
docs_tokens = None  # [doc.split() for doc in corpus]

# TAREFA DO ALUNO: Calcular TF (term frequency) para cada documento
# TF(t, d) = count(t in d) / total words in d

# TAREFA DO ALUNO: Calcular IDF para cada palavra
# IDF(t) = log(N / df(t)) onde N = num docs, df = num docs contendo t

# TAREFA DO ALUNO: Calcular TF-IDF = TF * IDF

# TAREFA DO ALUNO: Para cada documento, qual palavra tem maior TF-IDF?
print('Complete os TAREFA DO ALUNOs acima!')

In [ ]:
# SOLUCAO - Exercicio 2: TF-IDF Manual
corpus = [
    "gato gato cachorro",
    "cachorro cachorro cachorro gato",
    "passaro passaro passaro passaro"
]

docs_tokens = [doc.split() for doc in corpus]
all_words = sorted(set(w for doc in docs_tokens for w in doc))
N = len(corpus)

print('TF-IDF Manual - SOLUCAO')
print('=' * 60)
print()

# TF
print('1. Term Frequency (TF):')
tf_vals = {}
for i, doc in enumerate(docs_tokens):
    tf_vals[i] = {}
    total = len(doc)
    for w in all_words:
        count = doc.count(w)
        tf_vals[i][w] = count / total
        if count > 0:
            print(f'   Doc {i}: TF("{w}") = {count}/{total} = {tf_vals[i][w]:.3f}')
print()

# IDF
print('2. Inverse Document Frequency (IDF):')
idf_vals = {}
for w in all_words:
    df = sum(1 for doc in docs_tokens if w in doc)
    idf_vals[w] = math.log(N / df)
    print(f'   IDF("{w}") = log({N}/{df}) = {idf_vals[w]:.3f}')
print()

# TF-IDF
print('3. TF-IDF = TF * IDF:')
for i in range(N):
    best_word = ''
    best_score = 0
    for w in all_words:
        score = tf_vals[i][w] * idf_vals[w]
        if score > 0:
            print(f'   Doc {i}: TFIDF("{w}") = {tf_vals[i][w]:.3f} * {idf_vals[w]:.3f} = {score:.3f}')
            if score > best_score:
                best_score = score
                best_word = w
    print(f'   -> Mais informativa para Doc {i}: "{best_word}" ({best_score:.3f})')
    print()

print('Observe: "passaro" tem TF-IDF alto no Doc 2 porque so aparece la (IDF alto)')
print('"gato" e "cachorro" tem IDF mais baixo porque aparecem em 2 docs')

### Exercicio 3: Classificador de Sentimento

**Tarefa:** Usando a classe NaiveBayesText definida acima, avalie o classificador
em textos ambiguos e calcule a accuracy num conjunto de teste.

In [ ]:
# PRATICA - Exercicio 3: Classificador de Sentimento
# Use o NaiveBayesText ja treinado (nb_model) definido acima

test_data = [
    ("produto bom funciona bem", "positivo"),
    ("nao recomendo pessimo", "negativo"),
    ("excelente adorei perfeito", "positivo"),
    ("horrivel nao funciona", "negativo"),
    ("qualidade razoavel nada especial", "positivo"),
    ("terrivel defeituoso ruim", "negativo"),
]

# TAREFA DO ALUNO: Fazer predicoes para cada texto de teste
# TAREFA DO ALUNO: Calcular accuracy (corretas / total)
# TAREFA DO ALUNO: Mostrar quais o modelo errou e por que

correct = 0
total = len(test_data)
for text, true_label in test_data:
    pred = None  # nb_model.predict(text)[0]
    # TAREFA DO ALUNO: comparar pred com true_label

if pred is not None:
    print(f'Accuracy: {correct/total:.1%}')
else:
    print('Complete os TAREFA DO ALUNOs acima!')

In [ ]:
# SOLUCAO - Exercicio 3: Classificador de Sentimento
test_data = [
    ("produto bom funciona bem", "positivo"),
    ("nao recomendo pessimo", "negativo"),
    ("excelente adorei perfeito", "positivo"),
    ("horrivel nao funciona", "negativo"),
    ("qualidade razoavel nada especial", "positivo"),
    ("terrivel defeituoso ruim", "negativo"),
]

print('Avaliacao do Classificador Naive Bayes - SOLUCAO')
print('=' * 60)

correct = 0
total = len(test_data)
for text, true_label in test_data:
    pred, scores = nb_model.predict(text)
    is_correct = pred == true_label
    correct += int(is_correct)
    status = 'OK' if is_correct else 'ERRO'
    print(f'  [{status}] "{text}"')
    print(f'         True: {true_label}, Pred: {pred}')
    if not is_correct:
        print(f'         Motivo provavel: palavras ambiguas ou nao vistas no treino')

print()
print(f'Accuracy: {correct}/{total} = {correct/total:.1%}')
print()
if correct < total:
    print('Erros tipicos de Naive Bayes em sentimento:')
    print('  - Palavras nao vistas no treino recebem probabilidade uniforme')
    print('  - Nao captura negacoes ("nao bom" = neg, mas "bom" puxa para pos)')
    print('  - Assume independencia entre palavras')

### O que observar sobre Vocabulario e Cobertura

O vocabulario de um corpus tipico em portugues tem:
- 10K-50K palavras unicas em textos jornalisticos
- 100K-500K com nomes proprios e termos tecnicos
- 1M+ em corpora web (incluindo erros de digitacao, giriia)

BoW/TF-IDF com vocabulario de 50K features ja cobre ~95% das palavras.
Mas as 5% restantes (palavras raras) frequentemente sao as MAIS informativas.

### O que concluir sobre Texto como Dados de Alta Dimensionalidade

Texto e inerentemente de ALTA dimensionalidade. Um corpus com 100K palavras
unicas gera vetores de 100K dimensoes, a maioria zeros.
Isso e o "curse of dimensionality" para texto. Reducao de dimensionalidade
(PCA, SVD/LSA) pode ajudar, mas a abordagem moderna e usar embeddings densos (5B_2).

### Conexao com outros notebooks sobre Esparsidade e Eficiencia

A esparsidade de BoW conecta com 3_1 (k-means com features sparse) e 4_5
(aceleracao). Matrizes sparse tem operacoes otimizadas (scipy.sparse)
que permitem classificacao em millisegundos mesmo com milhoes de features.

### O que observar sobre Tokenizacao Subword

Tokenizacao classica (split por espacos) tem limitacoes:
- "correndo" e "correr" sao tokens diferentes
- Palavras novas (OOV) nao existem no vocabulario
- Linguagens aglutinantas (alemao, turco) geram vocabularios enormes

BPE (Byte Pair Encoding) resolve isso dividindo palavras em subwords:
"correndo" -> "corr" + "endo". Isso e usado em BERT, GPT, etc. (5B_4)

### O que concluir sobre a Importancia de Baselines em NLP

Na pratica industrial, sempre comece com TF-IDF + Logistic Regression como baseline.
Este combo simples frequentemente atinge 85-90% de accuracy em classificacao
de texto e e 100x mais rapido que fine-tuning de BERT. Documente este baseline
e so avance para modelos mais complexos se a melhoria justificar o custo.

### Conexao com outros notebooks sobre Validacao Experimental

Comparar NLP classico vs deep learning e um experimento que precisa de
design rigoroso (1_5): mesma split de treino/teste, mesmas metricas,
intervalo de confianca. Sem isso, nao da para afirmar que BERT e "melhor".

## 8. Erros Comuns e Armadilhas

### Erro 1: Nao Fazer Pre-processamento

**O que acontece:** "Casa" e "casa" sao features diferentes. Vocabulario explode.

**Por que em ML:** Modelos BoW/TF-IDF tratam cada string unica como feature separada.
Sem lowercase, o modelo tem o dobro de features para metade dos dados.

**Solucao:** Sempre normalizar (lowercase, remover pontuacao) antes de vetorizar.

### Erro 2: Ignorar Stopwords em Datasets Pequenos

**O que acontece:** Stopwords dominam o vocabulario e diluem o sinal.

**Por que em ML:** "o", "a", "de" aparecem em TAREFA DO ALUNOS os documentos (IDF ~0).
Mas ainda ocupam espaco no vetor e adicionam ruido.

**Solucao:** Remover stopwords para BoW/TF-IDF. Para deep learning (BERT), NaO remover.

### Erro 3: TF-IDF sem Normalizar Comprimento

**O que acontece:** Documentos longos tem scores TF-IDF maiores que curtos.

**Por que em ML:** TF conta frequencia absoluta. Um doc de 1000 palavras
naturalmente tem mais ocorrencias que um de 100.

**Solucao:** Usar TF normalizado (count / total_words) ou L2-normalizar os vetores.

### Erro 4: Usar BoW para Tarefas que Exigem Ordem

**O que acontece:** "nao gostei" e "gostei" tem BoW quase identicos.

**Por que em ML:** BoW ignora ordem. A negacao "nao" vira uma feature separada.

**Solucao:** Usar bigramas ("nao_gostei") ou modelos sequenciais (RNN, LSTM, BERT).

### Erro 5: Vocabulario Muito Grande sem Poda

**O que acontece:** Memoria estoura. Modelo lento. Features ruidosas.

**Por que em ML:** Palavras que aparecem 1 vez no corpus nao ajudam o modelo.

**Solucao:** Remover palavras com df < min_df (ex: 5) e df > max_df (ex: 95%).

### O que observar sobre Multilinguismo em NLP

NLP classico e MUITO sensivel ao idioma:
- Stopwords sao diferentes por idioma
- Stemming precisa de regras especificas
- Tokenizacao em chines/japones nao usa espacos
- Portuguese tem mais flexao que ingles

Ferramentas universais: byte-pair encoding (BPE) e subword tokenization
resolvem isso para deep learning, mas NLP classico ainda precisa de adaptacao manual.

### O que concluir sobre a Relevancia Atual de NLP Classico

Apesar de LLMs dominarem benchmarks, NLP classico ainda e usado quando:
1. Precisa de interpretabilidade (TF-IDF mostra quais palavras importam)
2. Latencia e critica (BoW + NB e ~1000x mais rapido que BERT)
3. Dados sao escassos (NB funciona com 100 exemplos, BERT precisa de 1000+)
4. Baseline: sempre compare com NLP classico antes de ir para deep learning

### Conexao com outros notebooks sobre Baseline e Complexidade

O principio de comecar com modelos simples (1_4) aplica-se fortemente em NLP.
BoW + Logistic Regression e o baseline classico. Se ele resolve o problema
com 90% accuracy, voce NAO precisa de BERT.

### O que observar sobre Avaliacao em NLP

Metricas para NLP sao as mesmas de classificacao (2_5):
- Accuracy, Precision, Recall, F1
- Macro vs Micro averaging para multiclass
- Confusion matrix para entender erros

Mas NLP tem metricas adicionais: BLEU (traducao), ROUGE (sumarizacao),
Perplexity (language models). Cada tarefa tem sua metrica principal.

### O que concluir sobre o Caminho de Aprendizado em NLP

Este notebook e a BASE. O caminho e:
1. NLP Classico (este notebook) -> entender representacoes e pre-processamento
2. Word Embeddings (5B_2) -> representacoes densas
3. RNNs/LSTMs (5B_3) -> sequencias
4. Transformers/BERT (5B_4) -> attention e pre-training
5. LLMs (5B_5) -> modelos massivos
6. RAG (5B_6) -> aplicacoes praticas

### Conexao com outros notebooks sobre o Modulo 5B

Cada conceito deste notebook reaparece nos seguintes:
- Pre-processamento -> tokenization em BPE (5B_4)
- BoW -> evolui para embeddings (5B_2)
- N-gramas -> evolui para attention (5B_4)
- Naive Bayes -> evolui para fine-tuned BERT (5B_4)

### O que concluir sobre Escalabilidade de NLP Classico

NLP classico escala BEM horizontalmente (mais documentos = rapido com sparse ops)
mas escala MAL verticalmente (mais vocabulario = mais features = mais esparsidade).
Deep learning escala bem verticalmente (embeddings de dimensao fixa) mas custa
mais compute por documento. A escolha depende do trade-off custo/accuracy do seu caso.

### Conexao com outros notebooks sobre Custo Computacional

O custo de inferencia conecta com 4_5 (aceleracao hardware). NLP classico
roda em CPU comum. BERT precisa de GPU. LLMs precisam de clusters.
Para aplicacoes de alto volume (milhoes de documentos/dia), NLP classico
ainda pode ser a unica opcao viavel economicamente.

## Resumo e Proximos Passos

### Hierarquia de Conceitos

```
NLP Classico
|
|-- Pre-processamento
|   |-- Tokenizacao (palavras, subwords, caracteres)
|   |-- Normalizacao (lowercase, acentos)
|   |-- Stopwords (remover palavras sem conteudo)
|   +-- Stemming/Lemmatization (reduzir formas)
|
|-- Representacoes Sparse
|   |-- Bag of Words (frequencia bruta)
|   |-- TF-IDF (frequencia x raridade)
|   +-- N-gramas (capturar sequencias)
|
|-- Classificacao de Texto
|   |-- Naive Bayes (baseline rapido)
|   |-- Logistic Regression (linear, interpretavel)
|   +-- SVM (margens, bom para texto)
|
+-- Analise de Sentimento
    |-- Baseada em lexico (dicionario)
    |-- Baseada em ML (BoW + classificador)
    +-- Desafios: negacao, ironia, dominio
```

### Conexoes entre Notebooks

| Conceito deste notebook | Conecta com | Relacao |
|------------------------|-------------|---------|
| Pre-processamento | 1_3 (feature engineering) | Feature extraction para texto |
| BoW/TF-IDF | 3_2 (reducao dimensionalidade) | Vetores sparse de alta dimensao |
| Naive Bayes | 2_1 (classificacao) | Mesmo algoritmo, features diferentes |
| Similaridade cosseno | 1_2 (metricas de distancia) | Metrica para vetores sparse |
| N-gramas | 5B_4 (attention) | Contexto local vs global |

### Checklist de Competencias

- [ ] Sei implementar pipeline de pre-processamento (tokenize, stopwords, stem)
- [ ] Entendo BoW e TF-IDF e suas diferencas
- [ ] Sei calcular TF-IDF manualmente
- [ ] Entendo por que Naive Bayes funciona bem para texto
- [ ] Sei quando usar NLP classico vs deep learning
- [ ] Entendo limitacoes de representacoes sparse

### Proximos Passos

- **5B_2:** Word Embeddings (representacoes DENSAS que capturam semantica)
- **5B_3:** RNNs e LSTMs (modelos SEQUENCIAIS para texto)
- **Pratica:** Aplicar pipeline em um dataset real de reviews